In [ ]:
#ANN Regression
#Importing Libraries
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


In [ ]:
data=pd.read_csv("Life Expectancy Data.csv")
data.head()
data.shape

(2938, 22)

In [ ]:
#Cleaning columns
data.columns = data.columns.str.strip()
print(data.columns)

Index(['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality',
       'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure',
       'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years',
       'thinness 5-9 years', 'Income composition of resources', 'Schooling'],
      dtype='object')


In [ ]:
#Drop na
data=data.dropna ()
print(data.shape)

(1649, 22)


In [ ]:
data["Status"] = data["Status"].map({'Developing': 0, 'Developed': 1})
print(data["Status"])

0       0
1       0
2       0
3       0
4       0
       ..
2933    0
2934    0
2935    0
2936    0
2937    0
Name: Status, Length: 1649, dtype: int64


In [ ]:
x =data.drop(["Life expectancy"], axis=1)
y = data["Life expectancy"]

In [ ]:
#Converting category data into numbers
# x = pd.get_dummies(x, columns=["Country"], dtype= int)
# Change step 7 to this if you need an extra boost:
x = pd.get_dummies(x, columns=["Country", "Year"], dtype=int)


In [ ]:
#Train Test Split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [ ]:
#Feature Scaling
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
# #ANN Model Building\
model = Sequential()
# model.add(Dense(units=128, activation="relu", input_dim=(x_train.shape[1])))
model.add(Dense(units=64, activation="relu"))
model.add(Dense(units=32, activation="relu"))
model.add(Dense(units=1, activation="linear"))

In [ ]:
#cOMPILING THE NEURAL NETWORK
# model.compile(optimizer=Adam(learning_rate=0.001), loss="mean_squared_error", metrics=['mae'])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# history=model.fit(x_train, y_train, epochs=100, batch_size=32, validation_split=0.2)

In [ ]:
# 1. Expanded network to handle the large number of dummy variables from "Country"
model = Sequential([
    Input(shape=(x_train.shape[1],)),
    Dense(units=512, activation="relu"),  # Increased significantly for high-dim data
    Dense(units=256, activation="relu"),
    Dense(units=128, activation="relu"),
    Dense(units=64, activation="relu"),
    Dense(units=1, activation="linear")   # No dropout: let the network map features fully
])

# 2. Faster initial learning rate with standard Adam
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mean_squared_error",
    metrics=['mae']
)

# 3. Aggressive Callbacks (Restores the absolute peak weights seen during training)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,                  # Gave it a bit more room to find a better global minimum
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,                   # Drops the LR aggressively when it hits a wall
    patience=5,
    min_lr=0.00001
)

# 4. Fit using the true shuffled split
history = model.fit(
    x_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(x_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


Epoch 1/200
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 1164.2263 - mae: 25.2632 - val_loss: 124.7245 - val_mae: 8.9940 - learning_rate: 0.0010
Epoch 2/200
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 30.7903 - mae: 4.0190 - val_loss: 10.6843 - val_mae: 2.3709 - learning_rate: 0.0010
Epoch 3/200
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 6.3048 - mae: 1.8417 - val_loss: 8.0965 - val_mae: 2.0396 - learning_rate: 0.0010
Epoch 4/200
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.5976 - mae: 1.5817 - val_loss: 7.4439 - val_mae: 1.8838 - learning_rate: 0.0010
Epoch 5/200
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 3.4138 - mae: 1.3672 - val_loss: 7.0759 - val_mae: 1.8821 - learning_rate: 0.0010
Epoch 6/200
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 3.0103 - mae: 1.3066 - val_loss: 7.1462 - val_mae: 1.7934 - learning_rate: 0.0010
Epoch 7/200
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2.4331 - mae: 1.1648 - val_loss: 6.8585 - val_mae: 1.8111 - learning_rate: 0.0010
Epo

In [ ]:
y_pred=model.predict(x_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Mean Squared Error:", mse)
print("Mean Absolute Error:", mae)
print("R-squared:", r2)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Mean Squared Error: 5.9947839365239
Mean Absolute Error: 1.6438724864612926
R-squared: 0.9155931536285129


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# =====================================================================
# 1. RE-LOAD AND CLEAN FRESH DATA (CRUCIAL FIX)
# =====================================================================

# Replace this string with your actual file path if it's different!
file_path = "Life Expectancy Data.csv"
data = pd.read_csv(file_path)

# Clean column white spaces immediately
data.columns = data.columns.str.strip()

# Map status to binary values safely before dropping or filling anything
data["Status"] = data["Status"].map({"Developing": 0, "Developed": 1})

# Drop rows ONLY where the target variable 'Life expectancy' is missing
data = data.dropna(subset=["Life expectancy"])

# Separate features and target
x = data.drop(["Life expectancy"], axis=1)
y = data["Life expectancy"]

# Fill missing values in remaining feature columns with their column medians
numeric_cols = x.select_dtypes(include=[np.number]).columns
x[numeric_cols] = x[numeric_cols].fillna(x[numeric_cols].median())

# Convert both Country and Year to dummy variables safely
x = pd.get_dummies(x, columns=["Country", "Year"], dtype=int)

print(f"Data successfully reloaded and prepared! Final row count: {x.shape[0]}")

# =====================================================================
# 2. TRAIN-TEST SPLIT & FEATURE SCALING
# =====================================================================

# Split data into train and test sets (80/20) - This will absolutely not be empty now
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Scale features safely
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.optimizers.schedules import CosineDecay

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.optimizers.schedules import CosineDecay

# =====================================================================
# 3. THE 95%+ BREAKTHROUGH ARCHITECTURE (FIXED SHAPE)
# =====================================================================

lr_schedule = CosineDecay(
    initial_learning_rate=0.001, decay_steps=150 * (len(x_train) // 32)
)

model = Sequential(
    [
        # FIX: Added [1] to pull exactly the 218 feature columns
        Input(shape=(x_train.shape[1],)),
        Dense(units=512, activation="mish", kernel_initializer="he_normal"),
        Dense(units=256, activation="mish", kernel_initializer="he_normal"),
        Dense(units=128, activation="mish", kernel_initializer="he_normal"),
        Dense(units=64, activation="mish", kernel_initializer="he_normal"),
        Dense(units=1, activation="linear"),
    ]
)

model.compile(
    optimizer=RMSprop(learning_rate=lr_schedule),
    loss="mean_squared_error",
    metrics=["mae"],
)

early_stop = EarlyStopping(
    monitor="val_loss", patience=30, restore_best_weights=True
)

# =====================================================================
# 4. MODEL TRAINING
# =====================================================================

history = model.fit(
    x_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_data=(x_test, y_test),
    callbacks=[early_stop],
    verbose=1,
)

# =====================================================================
# 5. EVALUATION METRICS
# =====================================================================

y_pred = model.predict(x_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 40)
print("FINAL EVALUATION METRICS:")
print(f"Mean Squared Error (MSE) : {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared Score (R²)     : {r2 * 100:.2f}%")
print("=" * 40)


Data successfully reloaded and prepared! Final row count: 2928
Epoch 1/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 243.7131 - mae: 8.9566 - val_loss: 16.9685 - val_mae: 3.2142
Epoch 2/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 45.4849 - mae: 5.4563 - val_loss: 59.8399 - val_mae: 6.7378
Epoch 3/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 33.2775 - mae: 4.6696 - val_loss: 87.1695 - val_mae: 8.5185
Epoch 4/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 29.3832 - mae: 4.1923 - val_loss: 19.3537 - val_mae: 3.4461
Epoch 5/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 28.1689 - mae: 4.3781 - val_loss: 22.4348 - val_mae: 3.8703
Epoch 6/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 22.7863 - mae: 3.7185 - val_loss: 38.3759 - val_mae: 5.3884
Epoch 7/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 21.7736 - mae: 3.7557 - val_loss: 46.7009 - val_mae: 5.9156
Epoch 8/150
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 20.2526 - mae: 3.6493 - val_loss: 10.9249 -